In [1]:
import os
import time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/data', force_remount=True)

ROOT_DIRECTORY = 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista'
SPLITS_CSV_PATH = ROOT_DIRECTORY + '/k-fold_split/typography_dataset_splits.csv'

Mounted at /content/data


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [4]:
class TypographyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'image_path']
        label = self.dataframe.loc[idx, 'label_encoded']
        
        # Carrega a imagem e garante que seja RGB (3 canais) para a ResNet
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Erro ao carregar a imagem {img_path}: {e}")
            # Retorna uma imagem preta em caso de erro para não quebrar o loop
            image = Image.new('RGB', (224, 224))
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [5]:
def load_fold_from_csv(csv_path, target_fold_idx, label_encoder):
    df = pd.read_csv(csv_path)
    
    # Transforma os rótulos de texto em números e adiciona ao DataFrame
    df['label_encoded'] = label_encoder.transform(df['label'])
    
    fold_df = df[df['fold'] == target_fold_idx]
    train_df = fold_df[fold_df['split_type'] == 'train']
    val_df = fold_df[fold_df['split_type'] == 'val']
    
    return train_df, val_df

In [6]:
# ==========================================
# 2. DEFINIÇÃO DAS TRANSFORMAÇÕES
# ==========================================
# SEM Data Augmentation - Apenas redimensionamento e normalização padrão do ImageNet
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [7]:
# ==========================================
# 3. LOOP DE TREINAMENTO DO FOLD
# ==========================================
def train_resnet_fold(train_loader, val_loader, num_classes=4, epochs=10):
    # Carrega a ResNet-18 pré-treinada
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    
    # Substitui a última camada para classificar nossas 4 classes em vez de 1000
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
    # Fase de Validação final do fold
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return all_labels, all_preds

In [8]:
# ==========================================
# 4. EXECUÇÃO DA VALIDAÇÃO CRUZADA
# ==========================================
def run_cv_resnet(csv_path, num_folds=5, batch_size=32, epochs=5):
    print("Iniciando Validação Cruzada com ResNet-18 (Sem Augmentation)...")
    
    classes = sorted(['grotesco', 'serifado', 'escritural', 'fantasia'])
    label_encoder = LabelEncoder()
    label_encoder.fit(classes)
    
    fold_metrics = []
    
    for fold_idx in range(num_folds):
        print(f"\n{'='*40}")
        print(f"Processando Fold {fold_idx + 1}/{num_folds} (ResNet-18)")
        print(f"{'='*40}")
        
        train_df, val_df = load_fold_from_csv(csv_path, fold_idx, label_encoder)
        
        train_dataset = TypographyDataset(train_df, transform=base_transform)
        val_dataset = TypographyDataset(val_df, transform=base_transform)
        
        # DataLoaders gerenciam o envio de lotes de imagens para o modelo
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        start_time = time.time()
        
        # Treinando com poucas épocas (5) para o teste inicial, pois a rede já é pré-treinada
        y_true, y_pred = train_resnet_fold(train_loader, val_loader, num_classes=4, epochs=epochs)
        
        training_time = time.time() - start_time
        fold_accuracy = accuracy_score(y_true, y_pred)
        
        print(f"Fold {fold_idx + 1} Acurácia: {fold_accuracy:.4f} (Tempo: {training_time:.2f}s)")
        print(classification_report(y_true, y_pred, labels=[0, 1, 2, 3], target_names=classes, zero_division=0))
        
        fold_metrics.append(fold_accuracy)
        
    avg_accuracy = np.mean(fold_metrics)
    std_accuracy = np.std(fold_metrics)
    
    print(f"\n{'='*40}")
    print("RESULTADO FINAL CROSS-VALIDATION - RESNET-18")
    print(f"Acurácia Média: {avg_accuracy:.4f} ± {std_accuracy:.4f}")
    print(f"{'='*40}\n")

In [9]:
if __name__ == "__main__":

    if os.path.exists(SPLITS_CSV_PATH):
        # Estamos usando 5 épocas apenas para estabelecer o baseline de DL mais rapidamente
        run_cv_resnet(SPLITS_CSV_PATH, num_folds=5, batch_size=32, epochs=10)
    else:
        print("Erro: CSV não encontrado.")

Iniciando Validação Cruzada com ResNet-18 (Sem Augmentation)...

Processando Fold 1/5 (ResNet-18)
Fold 1 Acurácia: 0.6321 (Tempo: 75.87s)
              precision    recall  f1-score   support

  escritural       0.91      0.72      0.80        94
    fantasia       0.00      0.00      0.00        11
    grotesco       0.47      0.85      0.61        54
    serifado       0.57      0.38      0.45        53

    accuracy                           0.63       212
   macro avg       0.49      0.49      0.47       212
weighted avg       0.67      0.63      0.63       212


Processando Fold 2/5 (ResNet-18)
Fold 2 Acurácia: 0.6547 (Tempo: 74.49s)
              precision    recall  f1-score   support

  escritural       0.67      1.00      0.80        49
    fantasia       1.00      0.10      0.17        21
    grotesco       0.66      0.77      0.71        90
    serifado       0.59      0.41      0.49        63

    accuracy                           0.65       223
   macro avg       0.73    

In [10]:
if __name__ == "__main__":

    if os.path.exists(SPLITS_CSV_PATH):
        # Estamos usando 5 épocas apenas para estabelecer o baseline de DL mais rapidamente
        run_cv_resnet(SPLITS_CSV_PATH, num_folds=5, batch_size=32, epochs=15)
    else:
        print("Erro: CSV não encontrado.")

Iniciando Validação Cruzada com ResNet-18 (Sem Augmentation)...

Processando Fold 1/5 (ResNet-18)
Fold 1 Acurácia: 0.6038 (Tempo: 110.41s)
              precision    recall  f1-score   support

  escritural       0.89      0.60      0.71        94
    fantasia       0.00      0.00      0.00        11
    grotesco       0.48      0.91      0.62        54
    serifado       0.62      0.43      0.51        53

    accuracy                           0.60       212
   macro avg       0.50      0.48      0.46       212
weighted avg       0.67      0.60      0.60       212


Processando Fold 2/5 (ResNet-18)
Fold 2 Acurácia: 0.6143 (Tempo: 109.97s)
              precision    recall  f1-score   support

  escritural       0.74      0.80      0.76        49
    fantasia       0.00      0.00      0.00        21
    grotesco       0.59      0.96      0.73        90
    serifado       0.50      0.19      0.28        63

    accuracy                           0.61       223
   macro avg       0.46  

In [ ]:
if __name__ == "__main__":

    if os.path.exists(SPLITS_CSV_PATH):
        # Estamos usando 5 épocas apenas para estabelecer o baseline de DL mais rapidamente
        run_cv_resnet(SPLITS_CSV_PATH, num_folds=5, batch_size=32, epochs=20)
    else:
        print("Erro: CSV não encontrado.")

Iniciando Validação Cruzada com ResNet-18 (Sem Augmentation)...

Processando Fold 1/5 (ResNet-18)
Fold 1 Acurácia: 0.6132 (Tempo: 145.77s)
              precision    recall  f1-score   support

  escritural       0.98      0.57      0.72        94
    fantasia       0.00      0.00      0.00        11
    grotesco       0.42      0.81      0.55        54
    serifado       0.64      0.60      0.62        53

    accuracy                           0.61       212
   macro avg       0.51      0.50      0.47       212
weighted avg       0.70      0.61      0.62       212


Processando Fold 2/5 (ResNet-18)
Fold 2 Acurácia: 0.6368 (Tempo: 145.67s)
              precision    recall  f1-score   support

  escritural       0.52      0.90      0.66        49
    fantasia       1.00      0.24      0.38        21
    grotesco       0.73      0.77      0.75        90
    serifado       0.62      0.38      0.47        63

    accuracy                           0.64       223
   macro avg       0.72  